In [14]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

In [2]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-3
DEVICE = torch.device('cpu')

TRAIN_DIR = r'D:\ScanO-assign-Apaar\dataset\train'
TEST_DIR = r'D:\ScanO-assign-Apaar\dataset\test'
OUTPUT_DIR = 'submission/outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/sample_outputs', exist_ok=True)

In [3]:
def build_df(root):
    paths = []
    labels = []

    for label, cls in enumerate(['normal', 'pneumonia']):
        cls_path = os.path.join(root, cls)

        for img in os.listdir(cls_path):
            paths.append(os.path.join(cls_path, img))
            labels.append(label)

    return pd.DataFrame({
        'image_path': paths,
        'label': labels
    })

train_df = build_df(TRAIN_DIR)

In [4]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [5]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
class XrayDataset(Dataset):
    def __init__(self, df, tfms):
        self.df = df
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row.image_path).convert('RGB')
        img = self.tfms(img)

        label = torch.tensor(row.label).long()

        return img, label

In [7]:
train_ds = XrayDataset(train_df, train_tfms)
val_ds = XrayDataset(val_df, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [8]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(512, 2)
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [12]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=1
)

In [9]:
best_acc = 0
best_loss = np.inf

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for imgs, labels in tqdm(train_loader):

        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()

    val_loss = 0

    preds = []
    trues = []

    with torch.no_grad():

        for imgs, labels in val_loader:

            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            p = outputs.argmax(1)

            preds.extend(p.cpu().numpy())
            trues.extend(labels.cpu().numpy())

    acc = accuracy_score(trues, preds)

    print(f'Epoch {epoch+1}')
    print(f'Train Loss: {train_loss/len(train_loader):.4f}')
    print(f'Val Loss: {val_loss/len(val_loader):.4f}')
    print(f'Val Acc : {acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_model_phase2.pth')

100%|██████████| 32/32 [01:43<00:00,  3.24s/it]


Epoch 1
Train Loss: 0.5479
Val Loss: 0.7909
Val Acc : 0.6172


100%|██████████| 32/32 [01:45<00:00,  3.28s/it]


Epoch 2
Train Loss: 0.3758
Val Loss: 0.4265
Val Acc : 0.8828


100%|██████████| 32/32 [01:34<00:00,  2.96s/it]


...

Epoch 13
Train Loss: 0.1943
Val Loss: 0.3256
Val Acc : 0.8906


100%|██████████| 32/32 [00:56<00:00,  1.77s/it]


Epoch 14
Train Loss: 0.1887
Val Loss: 0.4831
Val Acc : 0.8203


100%|██████████| 32/32 [00:55<00:00,  1.75s/it]


Epoch 15
Train Loss: 0.2128
Val Loss: 0.3045
Val Acc : 0.8828


In [15]:
# ── Training Loop ─────────────────────────────────────────────
best_loss  = float('inf')
best_acc   = 0.0
best_state = None
history    = []

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds         = outputs.argmax(dim=1)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    scheduler.step(epoch_loss)
    history.append((epoch, epoch_loss, epoch_acc))

    print(f'Epoch {epoch:02d} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

    if epoch_loss < best_loss:
        best_loss  = epoch_loss
        best_acc   = epoch_acc
        best_state = copy.deepcopy(model.state_dict())
        print(f'  --> Best model saved (loss={best_loss:.4f})')

elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed/60:.1f} min')

Epoch 1/15: 100%|██████████| 32/32 [01:41<00:00,  3.19s/it]


Epoch 01 | Loss: 0.1982 | Acc: 0.9316
  --> Best model saved (loss=0.1982)


Epoch 2/15: 100%|██████████| 32/32 [01:35<00:00,  2.98s/it]


Epoch 02 | Loss: 0.1651 | Acc: 0.9414
  --> Best model saved (loss=0.1651)


Epoch 3/15: 100%|██████████| 32/32 [01:32<00:00,  2.90s/it]


...

Epoch 13 | Loss: 0.0189 | Acc: 0.9961
  --> Best model saved (loss=0.0189)


Epoch 14/15: 100%|██████████| 32/32 [01:02<00:00,  1.95s/it]


Epoch 14 | Loss: 0.0187 | Acc: 0.9961
  --> Best model saved (loss=0.0187)


Epoch 15/15: 100%|██████████| 32/32 [01:03<00:00,  1.99s/it]

Epoch 15 | Loss: 0.0108 | Acc: 0.9980
  --> Best model saved (loss=0.0108)

Training complete in 21.6 min


In [16]:
# ── Save best model ───────────────────────────────────────────
save_path = os.path.join(OUTPUT_DIR, 'best_model_phase2.pth')
torch.save(best_state, save_path)
print(f'Saved: {save_path}')
print(f'Best train loss : {best_loss:.4f}')
print(f'Best train acc  : {best_acc:.4f}')

Saved: submission/outputs\best_model_phase2.pth
Best train loss : 0.0108
Best train acc  : 0.9980
